# Course 6: Geometric Deep Learning and Graphs
## Session 12: Graph learning in medicine and the capstone

**Goal:** predict missing drug-drug interactions with simple heuristics and with a GNN, using a correct split and a ranking metric.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) on small built-in or synthetic data. Cells labelled NOT RUN need a library, a dataset download or a GPU; run them on Kaggle or Colab.

### 1. A synthetic interaction graph
ogbl-ddi needs a download, so the runnable part uses a synthetic graph: drugs fall into classes, and drugs in the same or in related classes interact more often.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

def sbm(sizes, p_in, p_out, rng):
    """Stochastic block model: dense inside a block, sparse between blocks."""
    labels = np.repeat(np.arange(len(sizes)), sizes)
    n = len(labels)
    P = np.where(labels[:, None] == labels[None, :], p_in, p_out)
    A = np.triu(rng.random((n, n)) < P, 1)
    A = (A | A.T).astype(np.float32)
    return A, labels

def norm_adj(A):
    """GCN normalization: D^-1/2 (A + I) D^-1/2."""
    A = A + np.eye(len(A), dtype=A.dtype)
    d = A.sum(1)
    return A / np.sqrt(np.outer(d, d))

class GCN(nn.Module):
    def __init__(self, d_in, d_hid, d_out):
        super().__init__()
        self.l1, self.l2 = nn.Linear(d_in, d_hid), nn.Linear(d_hid, d_out)
    def forward(self, X, A_hat):
        H = F.relu(A_hat @ self.l1(X))
        return A_hat @ self.l2(H)

In [ ]:
from sklearn.metrics import roc_auc_score
rng = np.random.default_rng(0); torch.manual_seed(0)
n, k = 300, 6
cls = rng.integers(0, k, n)
B = np.full((k, k), 0.01) + np.eye(k) * 0.29; B[0, 1] = B[1, 0] = 0.15; B[2, 3] = B[3, 2] = 0.15     # related classes
A_full = np.triu(rng.random((n, n)) < B[cls][:, cls], 1)
edges = np.argwhere(A_full); rng.shuffle(edges)
n_test = len(edges) // 5
test_pos, train_pos = edges[:n_test], edges[n_test:]
A = np.zeros((n, n), dtype=np.float32); A[train_pos[:, 0], train_pos[:, 1]] = 1; A = np.maximum(A, A.T)   # the model sees training edges only

def sample_neg(m):
    out = []
    while len(out) < m:
        i, j = rng.integers(0, n, 2)
        if i < j and not A_full[i, j]: out.append((i, j))
    return np.array(out)
test_neg = sample_neg(1000)
print("drugs", n, "| training edges", len(train_pos), "| test edges", len(test_pos))

### 2. The metric: Hits@K
Rank each true test edge against many non-edges. Hits@K is the share of true edges scored above the K-th best non-edge. It is much stricter than AUC, and it is the metric OGB uses for this task.

In [ ]:
def evaluate(score):
    sp, sn = score(test_pos), score(test_neg)
    auc = roc_auc_score(np.r_[np.ones(len(sp)), np.zeros(len(sn))], np.r_[sp, sn])
    hits = float((sp > np.sort(sn)[-20]).mean())
    return auc, hits

### 3. Heuristics first
Common neighbours, and Adamic-Adar (shared neighbours count for more when they have few connections).

In [ ]:
deg = A.sum(1)
CN = A @ A
AA = A @ np.diag(1 / np.log(np.maximum(deg, 2))) @ A
res = {"common neighbours": evaluate(lambda e: CN[e[:, 0], e[:, 1]]), "Adamic-Adar": evaluate(lambda e: AA[e[:, 0], e[:, 1]])}

### 4. A GNN link predictor
A GCN encoder gives every drug an embedding. An MLP scores a pair from the product of the two embeddings. Negatives are resampled every epoch.

In [ ]:
A_hat = torch.tensor(norm_adj(A)); A_hat_res = A_hat
emb = nn.Parameter(torch.randn(n, 64) * 0.1)                      # no node features, so learn an embedding per drug
enc = GCN(64, 64, 64); dec = nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 1))
opt = torch.optim.Adam([emb, *enc.parameters(), *dec.parameters()], lr=0.01)
tp = torch.tensor(train_pos)
score_t = lambda Z, e: dec(Z[e[:, 0]] * Z[e[:, 1]]).squeeze(-1)
for epoch in range(200):
    Z = enc(emb, A_hat)
    neg = torch.tensor(sample_neg(len(tp)))
    loss = F.binary_cross_entropy_with_logits(torch.cat([score_t(Z, tp), score_t(Z, neg)]),
                                              torch.cat([torch.ones(len(tp)), torch.zeros(len(neg))]))
    opt.zero_grad(); loss.backward(); opt.step()
with torch.no_grad(): Z = enc(emb, A_hat)
res["GCN + MLP"] = evaluate(lambda e: score_t(Z, torch.tensor(e)).detach().numpy())
for name, (auc, hits) in res.items(): print(f"{name:18s} AUC {auc:.3f}   Hits@20 {hits:.3f}")
assert res["GCN + MLP"][0] > 0.8 and res["Adamic-Adar"][0] > 0.8

Notice two things. A simple heuristic is a strong baseline, so always report one. And Hits@20 is far lower than AUC suggests: on this noisy synthetic graph no method ranks many true edges above the 20 best non-edges.

### 5. The leak to avoid
If test edges are left in the graph used for message passing, the model is shown the answers.

In [ ]:
A_leak = np.maximum(A, np.maximum(A_full, A_full.T).astype(np.float32))       # test edges left in by mistake
one_hop = lambda M: (lambda e: 10 * M[e[:, 0], e[:, 1]] + (M @ M)[e[:, 0], e[:, 1]])   # "is there an edge" + common neighbours
print("clean graph: AUC %.3f Hits@20 %.3f" % evaluate(one_hop(A)))
print("leaky graph: AUC %.3f Hits@20 %.3f   <- too good to be true" % evaluate(one_hop(A_leak)))
assert evaluate(one_hop(A_leak))[0] > 0.99

### 6. ogbl-ddi (NOT RUN here)

In [ ]:
# NOT RUN: needs `pip install ogb torch_geometric` and a download (4,267 drugs, about 1.3 million interactions).
from ogb.linkproppred import PygLinkPropPredDataset, Evaluator
ds = PygLinkPropPredDataset("ogbl-ddi"); split = ds.get_edge_split(); graph = ds[0]      # graph holds training edges only
evaluator = Evaluator("ogbl-ddi")                                                         # official metric: Hits@20
# split["valid"]["edge"], split["valid"]["edge_neg"], split["test"]["edge"], split["test"]["edge_neg"]
# hits = evaluator.eval({"y_pred_pos": pos_scores, "y_pred_neg": neg_scores})["hits@20"]
# The split is by protein target, so test drugs act differently from training drugs. Compare your number with the OGB leaderboard.

### Capstone checklist (TODO)
1. One sentence: the question, and who would use the answer.
2. The graph: what are the nodes and edges, and where could label information leak through them?
3. Two baselines, one of them with no graph at all.
4. A split that matches how the model would be used (by time, by patient, or by drug class).
5. Mean and standard deviation over at least 3 seeds.
6. One limitation, stated plainly.